# Perturb-Seqr: Term Search for a Known Drug's Gene Sets → Drug Mimickers

Workflow:

1. **Term search**: search Perturb-Seqr for every gene set whose name relates to a known drug
   (`DRUG_NAME`), using the `filterTerm` parameter -- this is Perturb-Seqr's documented "term
   search" capability (direct metadata search of all dataset gene sets).
2. **Pair up experiments**: group the matches into up/down pairs that belong to the same
   underlying experiment (same dataset, cell line, dose, etc. -- everything except direction).
3. **Check each gene set pair for drug mimickers**: for each paired experiment, reconstruct its
   up/down signature and run the paired mimicker/reverser search across Perturb-Seqr's full
   database.
4. **Aggregate**: combine results across all of the drug's own experiments to see which other
   drugs consistently mimic it, not just in a single experiment.

Default example: **Metformin**.

In [1]:
!pip install -q requests pandas matplotlib numpy


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 120)

PERTURBSEQR_URL = "https://perturbseqr.maayanlab.cloud/graphql"

# ---- User input: known drug to search for ----
DRUG_NAME = "Metformin"  # change this to any drug name you expect to find in Perturb-Seqr
MAX_EXPERIMENTS = 3      # how many of the drug's own up/down experiment pairs to check for mimickers

print(f"Searching Perturb-Seqr for: {DRUG_NAME!r}")

Searching Perturb-Seqr for: 'Metformin'


## Reference gene panel

Perturb-Seqr's `get_overlap` query only returns the *overlap* between a supplied gene list and a
target gene set (not the target's full membership), so a comprehensive reference list is needed
to recover as much of each experiment's true signature as possible. We use the **`universe.tsv`**
gene list from the [MacArthur Lab `gene_lists` repository](https://github.com/macarthur-lab/gene_lists)
(Broad Institute) -- ~19,000 human protein-coding gene symbols.

In [3]:
GENE_UNIVERSE_URL = "https://raw.githubusercontent.com/macarthur-lab/gene_lists/master/lists/universe.tsv"

_universe_resp = requests.get(GENE_UNIVERSE_URL)
_universe_resp.raise_for_status()
REFERENCE_GENE_PANEL = [line.strip() for line in _universe_resp.text.splitlines() if line.strip()]

print(f"Reference panel size: {len(REFERENCE_GENE_PANEL)} genes (source: {GENE_UNIVERSE_URL})")

Reference panel size: 19194 genes (source: https://raw.githubusercontent.com/macarthur-lab/gene_lists/master/lists/universe.tsv)


## Core Perturb-Seqr functions

`enrich_perturbseqr_single_set` (with `filter_term` set) implements Perturb-Seqr's **term search**:
searching all dataset gene sets by name/metadata. This endpoint has no significance threshold
(unlike the paired query's `topN`/`pvalueLe`), so a term match is returned regardless of the
supplied gene list's overlap magnitude -- meaning the "genes" argument doesn't need to be
comprehensive for this step; only `filter_term` needs to match.

In [4]:
def parse_perturbation_term(term):
    """
    Perturb-Seqr packs the perturbation name together with metadata (cell line, tissue,
    dose, KO/KD/Mut label, dataset ID, gene set size, etc.) in the 'term' string, separated
    by '::'. The perturbation name is always the first '::'-delimited field; everything
    after is metadata. The direction ('up'/'down'), when present, is the last
    whitespace-separated token of the *last* '::' field.
    """
    fields = term.split("::")
    perturbation = fields[0].strip()
    last_field = fields[-1].strip() if fields else ""
    tokens = last_field.split(" ")
    direction = tokens[-1].lower() if len(tokens) > 1 and tokens[-1].lower() in ("up", "down") else None
    return perturbation, direction


def strip_direction(term: str) -> str:
    """Remove a trailing ' up'/' down' token from a term string, leaving the shared
    dataset/experiment key so up and down records from the same experiment can be matched."""
    for suffix in (" up", " down", " Up", " Down", " UP", " DOWN"):
        if term.endswith(suffix):
            return term[: -len(suffix)]
    return term


def enrich_perturbseqr_single_set(geneset: list, first: int = 100, filter_term: str = "", library_names=None):
    """Single gene-set enrichment against Perturb-Seqr. With filter_term set, this serves as
    Perturb-Seqr's term search: it restricts results to gene sets whose term matches filter_term,
    regardless of the supplied geneset's overlap magnitude (no significance threshold on this
    endpoint)."""
    variables = {
        "filterTerm": filter_term,
        "offset": 0,
        "first": first,
        "filterFda": False,
        "sortBy": "pvalue_up",
        "filterKo": False,
        "genes": geneset,
    }
    if library_names is not None:
        variables["libraryNames"] = library_names

    query = {
        "operationName": "EnrichmentQuery",
        "variables": variables,
        "query": """query EnrichmentQuery(
                        $genes: [String]!
                        $filterTerm: String = ""
                        $offset: Int = 0
                        $first: Int = 10
                        $filterFda: Boolean = false
                        $sortBy: String = ""
                        $filterKo: Boolean = false
                        $libraryNames: [String]
                        ) {
                        currentBackground {
                            enrich(
                            genes: $genes
                            filterTerm: $filterTerm
                            offset: $offset
                            first: $first
                            filterFda: $filterFda
                            sortby: $sortBy
                            filterKo: $filterKo
                            libraryNames: $libraryNames
                            ) {
                            nodes {
                                geneSetHash
                                pvalue
                                adjPvalue
                                oddsRatio
                                nOverlap
                                geneSets {
                                nodes {
                                    term
                                    id
                                    nGeneIds
                                    geneSetFdaCountsById {
                                    nodes {
                                        approved
                                        count
                                    }
                                    }
                                    library {
                                    name
                                    }
                                }
                                totalCount
                                }
                            }
                            totalCount
                            geneSetCount
                            consensusCount
                            consensus {
                                drug
                                oddsRatio
                                pvalue
                                adjPvalue
                                approved
                                countSignificant
                                countInsignificant
                                countUpSignificant
                                pvalueUp
                                adjPvalueUp
                                oddsRatioUp
                                pvalueDown
                                adjPvalueDown
                                oddsRatioDown
                                libraries
                            }
                            }
                        }
                        }
                        """,
    }

    response = requests.post(PERTURBSEQR_URL, json=query)
    if not response.ok:
        print(f"Perturb-Seqr request failed ({response.status_code}). Response body:")
        print(response.text[:2000])
    response.raise_for_status()
    res = response.json()

    consensus = res["data"]["currentBackground"]["enrich"]["consensus"]
    enrichment = res["data"]["currentBackground"]["enrich"]["nodes"]
    df_consensus = pd.DataFrame(consensus).rename(columns={"drug": "perturbation"})

    df_enrichment = pd.json_normalize(
        enrichment,
        record_path=["geneSets", "nodes"],
        meta=["geneSetHash", "pvalue", "adjPvalue", "oddsRatio", "nOverlap"],
    )
    if df_enrichment.empty:
        return pd.DataFrame(), df_consensus

    df_enrichment["approved"] = df_enrichment["geneSetFdaCountsById.nodes"].map(
        lambda x: x[0]["approved"] if len(x) > 0 else False
    )
    df_enrichment["count"] = df_enrichment["geneSetFdaCountsById.nodes"].map(
        lambda x: x[0]["count"] if len(x) > 0 else 0
    )
    df_enrichment = df_enrichment.drop(columns=["geneSetFdaCountsById.nodes"])
    df_enrichment["perturbation"] = df_enrichment["term"].map(lambda t: parse_perturbation_term(t)[0])
    df_enrichment["direction"] = df_enrichment["term"].map(lambda t: parse_perturbation_term(t)[1])
    df_enrichment["dataset_key"] = df_enrichment["term"].map(strip_direction)
    df_enrichment = df_enrichment.rename(columns={"library.name": "library"})

    return df_enrichment, df_consensus


def get_overlap(genes: list, gene_set_id: str):
    """Return the subset of `genes` that overlap with the given Perturb-Seqr gene set."""
    query = {
        "operationName": "OverlapQuery",
        "variables": {"id": gene_set_id, "genes": genes},
        "query": """query OverlapQuery($id: UUID!, $genes: [String]!) {geneSet(id: $id) {
    overlap(genes: $genes) {
      nodes {
        symbol
        ncbiGeneId
        description
        summary
      }   }}}""",
    }
    response = requests.post(PERTURBSEQR_URL, json=query)
    if not response.ok:
        print(f"Overlap request failed ({response.status_code}). Response body:")
        print(response.text[:2000])
    response.raise_for_status()
    res = response.json()
    return [item["symbol"] for item in res["data"]["geneSet"]["overlap"]["nodes"]]


def get_overlap_chunked(genes: list, gene_set_id: str, chunk_size: int = 300):
    """Same as get_overlap, but splits `genes` into request-size-safe chunks and unions the
    results, since sending all ~19,000 reference genes in one request causes a 413 Payload Too
    Large error from the server."""
    recovered = set()
    for start in range(0, len(genes), chunk_size):
        chunk = genes[start:start + chunk_size]
        try:
            recovered.update(get_overlap(chunk, gene_set_id))
        except requests.HTTPError as e:
            print(f"  Chunk {start}-{start + len(chunk)} failed ({e}); skipping this chunk.")
    return sorted(recovered)

## Step 1: Term search for gene sets relating to the known drug

Searches for every gene set whose term matches `DRUG_NAME`. This endpoint is fundamentally an
overlap-based enrichment computation, not a plain metadata search -- a gene set only appears in
the results if it has **nonzero overlap** with the supplied `genes` list, even when `filterTerm`
matches its name. So the search is run once per chunk of the full reference gene universe
(unioning results across chunks), rather than with a small placeholder list, to guarantee we
find the drug's records regardless of which specific genes happen to make up its true signature.

Two refinements based on what the first run of this notebook actually returned:
- `first` is set much higher per chunk (3000, not 200) -- otherwise a genuine match can be
  pushed out of the returned window by many *other* unrelated gene sets that also happened to
  overlap with that same chunk, silently dropping true positives. This makes each chunk request
  slower/larger, so this step will take noticeably longer to run.
- `filterTerm` turned out to match broader study-level metadata rather than strictly the
  perturbation's own name -- e.g. searching "Metformin" also returned an unrelated gene
  knockdown record from the same underlying GEO series. Results are post-filtered to rows whose
  own parsed perturbation name actually equals `DRUG_NAME` (case-insensitive exact match -- this
  won't catch differently-named salt/formulation variants like "Metformin hydrochloride"; loosen
  the filter if you need those too).

In [5]:
def term_search_chunked(filter_term: str, gene_universe: list, chunk_size: int = 300, first: int = 3000):
    """Search for gene sets matching `filter_term`, chunking `gene_universe` across multiple
    requests so that at least one chunk is likely to have nonzero overlap with the target's
    true signature (this endpoint only returns gene sets with nonzero overlap with the supplied
    genes, regardless of filterTerm matching). Results are unioned and de-duplicated by id.

    `first` is set much higher than a typical single query (3000, vs. 200 previously) because
    each chunk's results compete with every other gene set that also has nonzero overlap with
    that chunk -- with a low `first`, a genuine match can rank outside the returned window and
    be silently dropped, purely because many *other* unrelated gene sets happened to overlap
    with the same chunk more strongly. This mirrors the same crowding-out issue seen earlier
    with the paired enrichment endpoint.

    Note: `filterTerm` appears to match broader study-level metadata rather than strictly the
    perturbation's own name -- e.g. searching "Metformin" can also return an unrelated gene
    knockdown record from the *same* underlying GEO series. The caller should post-filter
    results to rows whose own parsed `perturbation` name actually matches (see below)."""
    all_matches = []
    n_chunks = (len(gene_universe) + chunk_size - 1) // chunk_size
    for i, start in enumerate(range(0, len(gene_universe), chunk_size)):
        chunk = gene_universe[start:start + chunk_size]
        try:
            df_chunk, _ = enrich_perturbseqr_single_set(chunk, first=first, filter_term=filter_term)
        except requests.HTTPError as e:
            print(f"  Chunk {i + 1}/{n_chunks} failed ({e}); skipping this chunk.")
            continue
        if not df_chunk.empty:
            all_matches.append(df_chunk)

    if not all_matches:
        return pd.DataFrame()

    df_all = pd.concat(all_matches, ignore_index=True)
    df_all = df_all.drop_duplicates(subset=["id"], keep="first").reset_index(drop=True)
    return df_all


df_drug_search_raw = term_search_chunked(DRUG_NAME, REFERENCE_GENE_PANEL, chunk_size=300, first=3000)
print(f"Found {len(df_drug_search_raw)} raw gene-set record(s) with {DRUG_NAME!r} anywhere in matched study metadata")

# Post-filter: keep only rows whose own parsed perturbation name actually matches DRUG_NAME,
# dropping same-study-but-different-perturbation false positives (e.g. a gene knockdown record
# that just happens to come from the same GEO series as a real Metformin comparison).
if not df_drug_search_raw.empty:
    df_drug_search = df_drug_search_raw[
        df_drug_search_raw["perturbation"].str.lower() == DRUG_NAME.lower()
    ].reset_index(drop=True)
else:
    df_drug_search = df_drug_search_raw

print(f"Found {len(df_drug_search)} gene-set record(s) whose own perturbation name matches {DRUG_NAME!r}")
df_drug_search[["perturbation", "direction", "library", "dataset_key", "id", "nGeneIds", "approved"]] if not df_drug_search.empty else df_drug_search

Found 64 raw gene-set record(s) with 'Metformin' anywhere in matched study metadata
Found 58 gene-set record(s) whose own perturbation name matches 'Metformin'


,perturbation,direction,library,dataset_key,id,nGeneIds,approved
0,metformin,up,RummaGEO Chem,metformin::::::::::mouse::GSE179531::1::0,7e3609af-8028-46e3-9c3e-13ee0fe1d726,1592,True
1,Metformin,down,CREEDS Chem,Metformin::::::::::mus musculus::GSE35961.3308,7b8ed209-7663-4767-8648-0877b189c3a5,225,True
2,metformin,down,RummaGEO Chem,"metformin::::::::::mouse::GSE157049,GSE157051::6::2",d172d39f-370a-4c07-ad15-2dbfe0f98cc4,309,True
3,metformin,up,RummaGEO Chem,metformin::::::::::human::GSE146982::7::6,43147710-e0b5-46a1-a2a3-16da6a04872e,1306,True
4,metformin,up,LINCS L1000 CP,metformin::A375::::24H::0.125uM::REP.A024::P17::BRD-K79602928,1edb73b1-b686-48d7-87ea-91b2fc44fe7a,248,True
5,metformin,up,LINCS L1000 CP,metformin::MCF7::::24H::1.11uM::POL001::I09::BRD-K79602928,5911037e-0b1b-4808-8734-9d716ae96efa,247,True
6,metformin,down,LINCS L1000 CP,metformin::HA1E::::24H::0.125uM::REP.A024::P17::BRD-K79602928,f85cc443-c575-478e-8388-21101a5e8113,247,True
7,metformin,down,RummaGEO Chem,metformin::::::::::mouse::GSE134191::3::1,91ef3e16-72e8-4643-b851-d9893f4d4d9d,982,True
8,metformin,up,RummaGEO Chem,metformin::::::::::human::GSE247159::8::3,0d44c0b6-4170-4065-937f-11793d65d841,316,True
9,metformin,up,RummaGEO Chem,metformin::::::::::mouse::GSE134191::3::1,a2cb8429-0969-4ec9-a766-458de67e7763,499,True


## Step 2: Pair up matching gene sets into up/down experiments

Groups the term-search hits by `dataset_key` (the shared experiment identifier once the
up/down direction suffix is stripped) so we only keep pairs that have **both** an "up" and a
"down" record from the same experiment. The pairs with the most member genes (a rough proxy for
data quality/completeness) are prioritized, and up to `MAX_EXPERIMENTS` are kept.

In [6]:
if df_drug_search.empty:
    raise ValueError(
        f"No Perturb-Seqr records matched {DRUG_NAME!r}. Try a different spelling/salt form "
        "(e.g. 'Metformin hydrochloride' vs 'Metformin'), or check the drug is actually present "
        "in Perturb-Seqr's 16 gene-set libraries."
    )

paired_experiments = []
for dataset_key, group in df_drug_search.groupby("dataset_key"):
    up_rows = group[group["direction"] == "up"]
    down_rows = group[group["direction"] == "down"]
    if up_rows.empty or down_rows.empty:
        continue
    up_row = up_rows.iloc[0]
    down_row = down_rows.iloc[0]
    paired_experiments.append({
        "dataset_key": dataset_key,
        "library": up_row["library"],
        "up_id": up_row["id"],
        "down_id": down_row["id"],
        "up_nGeneIds": up_row["nGeneIds"],
        "down_nGeneIds": down_row["nGeneIds"],
    })

if not paired_experiments:
    raise ValueError(
        f"Found {DRUG_NAME!r} records, but none had both an 'up' and a 'down' direction record "
        "from the same experiment to pair into a signature. Inspect df_drug_search above."
    )

df_paired_experiments = pd.DataFrame(paired_experiments)
df_paired_experiments["total_genes"] = df_paired_experiments["up_nGeneIds"] + df_paired_experiments["down_nGeneIds"]
df_paired_experiments = df_paired_experiments.sort_values("total_genes", ascending=False).reset_index(drop=True)

print(f"Found {len(df_paired_experiments)} paired up/down experiment(s) for {DRUG_NAME!r}")
selected_experiments = df_paired_experiments.head(MAX_EXPERIMENTS)
print(f"Checking the top {len(selected_experiments)} (by total gene-set size) for drug mimickers:")
selected_experiments

Found 16 paired up/down experiment(s) for 'Metformin'
Checking the top 3 (by total gene-set size) for drug mimickers:


,dataset_key,library,up_id,down_id,up_nGeneIds,down_nGeneIds,total_genes
0,metformin::::::::::mouse::GSE179531::1::0,RummaGEO Chem,7e3609af-8028-46e3-9c3e-13ee0fe1d726,1c907d14-131c-4233-80f9-a6d2811b1e5b,1592,1465,3057
1,metformin::::::::::human::GSE146982::3::6,RummaGEO Chem,f6263516-d45e-4aae-86a3-16bf1805b9e3,1aa2f334-445c-48e8-835c-ebe5feecc454,1580,1359,2939
2,metformin::::::::::human::GSE146982::3::1,RummaGEO Chem,296f64e2-2077-46b2-aea4-6859ef66b721,28d2ff6e-a990-4325-b444-50f050570fb6,1188,1126,2314


## Step 3: For each paired experiment, reconstruct its signature and find drug mimickers

For each selected experiment: reconstruct `genes_up`/`genes_down` via `get_overlap_chunked`, then
run the same validated paired mimicker/reverser search used in earlier notebooks (one query
sorted by `pvalue_mimic` for trustworthy mimickers, one with `genes_up`/`genes_down` swapped --
also sorted by `pvalue_mimic` -- for trustworthy reversers, since there's no verified `sort_by`
value for reverse p-value directly).

In [ ]:
def enrich_perturbseqr_up_down(genes_up: list, genes_down: list, first: int = 5000, topN: int = 5000, sort_by: str = "pvalue_mimic"):
    """Paired up/down gene set enrichment against Perturb-Seqr (validated GraphQL schema)."""
    query = {
        "operationName": "PairEnrichmentQuery",
        "variables": {
            "filterTerm": "",
            "offset": 0,
            "first": first,
            "filterFda": False,
            "sortBy": sort_by,
            "filterKo": False,
            "topN": topN,
            "pvalueLe": 0.05,
            "genesUp": genes_up,
            "genesDown": genes_down,
        },
        "query": """query PairEnrichmentQuery($genesUp: [String]!, $genesDown: [String]!, $filterTerm: String = \"\", $offset: Int = 0, $first: Int = 10, $filterFda: Boolean = false, $sortBy: String = \"\", $filterKo: Boolean = false, $topN: Int = 10000, $pvalueLe: Float = 0.05) {
  currentBackground {
    pairedEnrich(
      filterTerm: $filterTerm
      offset: $offset
      first: $first
      filterFda: $filterFda
      sortby: $sortBy
      filterKo: $filterKo
      topN: $topN
      pvalueLe: $pvalueLe
      genesDown: $genesDown
      genesUp: $genesUp
      ) {
        totalCount
        consensusCount
        consensus {
          drug
          oddsRatio
          pvalue
          adjPvalue
          approved
          countSignificant
          countInsignificant
          countUpSignificant
          pvalueUp
          adjPvalueUp
          oddsRatioUp
          pvalueDown
          adjPvalueDown
          oddsRatioDown
          libraries
          }
          nodes {
            adjPvalueMimic
            adjPvalueReverse
            mimickerOverlap
            oddsRatioMimic
            oddsRatioReverse
            pvalueMimic
            pvalueReverse
            reverserOverlap
            geneSet {
              nodes {
                id
                nGeneIds
                term
                geneSetFdaCountsById {
                  nodes {
                    count
                    approved
                    }
                  }
                library {
                  name
                  }
                }
              }
            }
          }
        }
      }
""",
    }

    response = requests.post(PERTURBSEQR_URL, json=query)
    if not response.ok:
        print(f"Perturb-Seqr request failed ({response.status_code}). Response body:")
        print(response.text[:2000])
    response.raise_for_status()
    res = response.json()

    consensus = res["data"]["currentBackground"]["pairedEnrich"]["consensus"]
    enrichment = res["data"]["currentBackground"]["pairedEnrich"]["nodes"]

    df_consensus_pair = pd.DataFrame(consensus).rename(
        columns={
            "drug": "perturbation",
            "pvalueUp": "pvalueMimic",
            "pvalueDown": "pvalueReverse",
            "adjPvalueUp": "adjPvalueMimic",
            "adjPvalueDown": "adjPvalueReverse",
            "oddsRatioUp": "oddsRatioMimic",
            "oddsRatioDown": "oddsRatioReverse",
        }
    )

    if not enrichment:
        return pd.DataFrame(), df_consensus_pair

    df_enrichment_pair = pd.DataFrame(enrichment)
    df_enrichment_pair["term"] = df_enrichment_pair["geneSet"].map(lambda t: parse_perturbation_term(t["nodes"][0]["term"])[0])
    df_enrichment_pair["approved"] = df_enrichment_pair["geneSet"].map(
        lambda t: t["nodes"][0]["geneSetFdaCountsById"]["nodes"][0]["approved"]
        if t["nodes"][0]["geneSetFdaCountsById"]["nodes"]
        else False
    )
    df_enrichment_pair["library"] = df_enrichment_pair["geneSet"].map(lambda t: t["nodes"][0]["library"]["name"])
    df_enrichment_pair["geneSetIdUp"] = df_enrichment_pair["geneSet"].map(
        lambda t: next((node["id"] for node in t["nodes"] if " up" in node["term"]), None)
    )
    df_enrichment_pair["geneSetIdDown"] = df_enrichment_pair["geneSet"].map(
        lambda t: next((node["id"] for node in t["nodes"] if " down" in node["term"]), None)
    )

    df_enrichment_pair = df_enrichment_pair.drop(columns=["geneSet"])

    return df_enrichment_pair, df_consensus_pair


DRUG_KEYWORDS = [
    "l1000 chemical", "l1000 cp", "tahoe-100m", "nibr drug-seq", "microarrays cmap",
    "ginkgo bioworks", "sciplex", "deepcover moa", "creeds drug", "creeds chem",
    "rummageo drug", "rummageo chem", "rgeo chem",
]
GENE_KEYWORDS = [
    "perturb atlas", "patlas", "l1000 gene knockouts", "l1000 xpr", "replogle",
    "creeds gene", "cm4ai", "rummageo gene", "rgeo gene",
]


def classify_library(lib: str) -> str:
    lib = str(lib).strip().lower()
    is_drug = any(kw in lib for kw in DRUG_KEYWORDS)
    is_gene = any(kw in lib for kw in GENE_KEYWORDS)
    if is_drug and is_gene:
        return "drug+gene"
    if is_drug:
        return "drug"
    if is_gene:
        return "gene"
    return "unknown"


def classify_libraries(libs) -> str:
    """Classify a consensus row that may span multiple libraries -- checks all of them
    rather than just the first, so a perturbation isn't misclassified just because an
    unrecognized library name happens to be listed first."""
    labels = {classify_library(lib) for lib in (libs or [])}
    if "drug" in labels and "gene" in labels:
        return "drug+gene"
    if "drug" in labels or "drug+gene" in labels:
        return "drug"
    if "gene" in labels:
        return "gene"
    return "unknown"


TOP_N = 10
experiment_results = {}

for _, exp in selected_experiments.iterrows():
    print(f"\n=== Experiment: {exp['dataset_key']}  (library={exp['library']}) ===")
    genes_up = get_overlap_chunked(REFERENCE_GENE_PANEL, exp["up_id"])
    genes_down = get_overlap_chunked(REFERENCE_GENE_PANEL, exp["down_id"])
    print(f"Reconstructed {len(genes_up)} up genes, {len(genes_down)} down genes")

    if len(genes_up) < 3 or len(genes_down) < 3:
        print("Too few genes recovered for a reliable mimicker search -- skipping this experiment.")
        continue

    df_enrichment_mimic, df_consensus_pair = enrich_perturbseqr_up_down(genes_up, genes_down, sort_by="pvalue_mimic")

    if df_consensus_pair.empty:
        print("No consensus perturbations found for this experiment's signature.")
        continue

    df_consensus_pair["perturbation_type"] = df_consensus_pair["libraries"].map(classify_libraries)
    top_drug_mimickers = (
        df_consensus_pair[df_consensus_pair["perturbation_type"] == "drug"]
        .sort_values("pvalueMimic")
        .head(TOP_N)
    )
    experiment_results[exp["dataset_key"]] = top_drug_mimickers

    print(f"Top {len(top_drug_mimickers)} drug mimickers for this experiment:")
    display(top_drug_mimickers[["perturbation", "pvalueMimic", "adjPvalueMimic", "oddsRatioMimic", "approved"]])

## Step 4: Aggregate across experiments

Since `DRUG_NAME` may have multiple independent experimental records in Perturb-Seqr (different
cell lines, doses, timepoints, or datasets), a drug mimicker that shows up **consistently across
several of the drug's own experiments** is a stronger signal than one that only appears in a
single experiment. This counts how often each candidate mimicker recurs.

In [ ]:
if experiment_results:
    all_hits = pd.concat(
        [df.assign(source_experiment=key) for key, df in experiment_results.items()],
        ignore_index=True,
    )
    mimicker_summary = (
        all_hits.groupby("perturbation")
        .agg(
            n_experiments=("source_experiment", "nunique"),
            best_pvalueMimic=("pvalueMimic", "min"),
            mean_oddsRatioMimic=("oddsRatioMimic", "mean"),
        )
        .sort_values(["n_experiments", "best_pvalueMimic"], ascending=[False, True])
    )
    print(f"Drug mimickers of {DRUG_NAME}, ranked by how many of its own experiments they showed up in:")
    display(mimicker_summary.head(20))
else:
    print("No experiment produced usable results to aggregate.")

## Summary

1. **Term search** (`filterTerm`) found all Perturb-Seqr gene sets relating to **`DRUG_NAME`**.
2. Paired matching records into complete up/down **experiments** (same dataset, different
   direction), prioritizing the most complete ones.
3. For each of the top `MAX_EXPERIMENTS` experiments, reconstructed its signature and ran the
   paired mimicker search to find **drug mimickers**.
4. Aggregated results across experiments to surface mimickers that recur consistently, not just
   in a single run.

Caveats:
- Signature reconstruction is still bounded by the reference gene universe's coverage.
- If `DRUG_NAME` has very few or no paired up/down experiments in Perturb-Seqr, this workflow
  won't find anything to check -- inspect `df_drug_search` to see what was actually matched.